In [1]:
from google.colab import files
uploaded = files.upload()

Saving df_processed.parquet to df_processed.parquet


In [2]:
import pandas as pd

df = pd.read_parquet('df_processed.parquet')
df.head()

,title,clause_type,question,is_impossible,processed
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,Highlight the parts (if any) of this contract ...,False,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,..."
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Highlight the parts (if any) of this contract ...,False,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,..."
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Agreement Date,Highlight the parts (if any) of this contract ...,False,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,..."
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Effective Date,Highlight the parts (if any) of this contract ...,False,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,..."
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Expiration Date,Highlight the parts (if any) of this contract ...,False,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,..."


In [3]:
from transformers import DistilBertTokenizerFast, DistilBertForQuestionAnswering
import torch

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertForQuestionAnswering.from_pretrained('distilbert-base-uncased')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [5]:
device = torch.device('cuda')
model.to(device)

DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

In [6]:
from torch.utils.data import Dataset

class CUADDataset(Dataset):

  def __init__(self, processed):
    self.processed = processed

  def __len__(self):
    return len(self.processed)

  def __getitem__(self, idx):
      processed = self.processed[idx]
      input_ids = list(processed['input_ids'])[:512]
      attention_mask = list(processed['attention_mask'])[:512]
      input_ids = torch.tensor(input_ids + [0] * (512 - len(input_ids)))
      attention_mask = torch.tensor(attention_mask + [0] * (512 - len(attention_mask)))
      start_position = torch.tensor(processed['start_position'])
      end_position = torch.tensor(processed['end_position'])

      return input_ids, attention_mask, start_position, end_position


In [7]:
train_dataset = CUADDataset(df['processed'].values)

In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [9]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

In [10]:
from tqdm import tqdm

epochs = 3
losses = []

model.train()
for epoch in range(epochs):
    epoch_loss = 0
    loop = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch}")
    for batch_idx, (input_ids, attention_mask, start_position, end_position) in loop:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        start_position = start_position.to(device)
        end_position = end_position.to(device)

        outputs = model(input_ids, attention_mask=attention_mask, start_positions=start_position, end_positions=end_position)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        loop.set_postfix(loss=round(loss.item(), 4))

    avg_loss = round(epoch_loss / len(train_loader), 4)
    losses.append(avg_loss)
    print(f"Epoch {epoch} | Avg Loss: {avg_loss}")

Epoch 0: 100%|██████████| 1307/1307 [19:36<00:00,  1.11it/s, loss=0.0151]


Epoch 0 | Avg Loss: 0.372


Epoch 1: 100%|██████████| 1307/1307 [20:04<00:00,  1.09it/s, loss=0.0146]


Epoch 1 | Avg Loss: 0.1887


Epoch 2: 100%|██████████| 1307/1307 [20:04<00:00,  1.08it/s, loss=0.202]

Epoch 2 | Avg Loss: 0.1418


In [11]:
model.save_pretrained('./lexiq_model')
tokenizer.save_pretrained('./lexiq_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./lexiq_model/tokenizer_config.json', './lexiq_model/tokenizer.json')

In [12]:
from google.colab import files
import shutil

shutil.make_archive('lexiq_model', 'zip', './lexiq_model')
files.download('lexiq_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>